In [0]:
import numpy as np
import pandas as pd

In [0]:
df = pd.read_csv('Data Climate/Data Climate Processed/processed_validated_stations_filled_gap30.csv')
df['Data Medicao'] = pd.to_datetime(df['Data Medicao'])
df

In [0]:
igrs = df['igr'].unique()
igrs

##### 1.1 Lag

- lag_1_semana_precipitacao (soma da semana anterior para precipitacao)
- lag_2_semanas_precipitacao (soma da semana duas semanas atrás para precipitacao)
- lag_3_semanas_precipitacao (soma da semana três semanas atrás para precipitacao)
- lag_4s_semanas_precipitacao (soma da semana quatro semanas atrás para precipitacao)
- lag_1_semana_umidade (média da semana anterior para umidade)
- lag_2_semanas_umidade (média da semana duas semanas atrás para umidade)
- lag_3_semanas_umidade (média da semana três semanas atrás para umidade)
- lag_4_semanas_umidade (média da semana quatro semanas atrás para umidade)
- lag_1_semana_temperatura (média da semana anterior para temperatura)
- lag_2_semanas_temperatura (média da semana duas semanas atrás para temperatura)
- lag_3_semanas_temperatura (média da semana três semanas atrás para temperatura)
- lag_4_semanas_temperatura (média da semana quatro semanas atrás para temperatura)

In [0]:
def create_lag(column, reference):
    global df, igrs

    for lag in [1, 2, 3, 4]:
        new_column_name = f'lag_{lag}_{reference}'
        
        # Initialize the column with NaNs only once
        if new_column_name not in df.columns:
            df[new_column_name] = np.nan

        for igr_i in igrs:
            mask = df['igr'] == igr_i
            df.loc[mask, new_column_name] = df.loc[mask, column].shift(lag)

In [0]:
create_lag(column='precipitation_weekly_sum', reference='precipitation')
create_lag(column='temp_mean_weekly_mean', reference='temp_mean')
create_lag(column='humidity_weekly_mean', reference='humidity')

In [0]:
# Check
df

##### 1.2 Change Between Weeks

- change_temperature (0 or 1 - percentile criterion between 0.75 and 0.85, when the previous week was below 0.75)
- change_precipitation (0 or 1 - criterion above 10mm accumulated rainfall)
- change_humidity (0 or 1 - percentile criterion between 0.75 and 0.85, when the previous week was below 0.75)

In [0]:
def is_between(a, x, b):
    return min(a, b) <= x <= max(a, b)

In [0]:
def create_variable_change(column, reference):
    global df, igrs

    new_column_name = f'change_{reference}'

    auxiliary_percentile_column = f'percentile_{column}'
    auxiliary_percentile_lag_column = f'percentile_lag_{column}'

        
    # Initialize the column with NaNs only once
    if new_column_name not in df.columns:
        df[new_column_name] = np.nan
        df[auxiliary_percentile_column] = np.nan

    for igr_i in igrs:
        mask = df['igr'] == igr_i

        # Calculate the percentile
        df.loc[mask, auxiliary_percentile_column] = df.loc[mask, column].rank(pct=True)
        df.loc[mask, auxiliary_percentile_lag_column] = df.loc[mask, auxiliary_percentile_column].shift(1)

    # Rules
    df[f'change_{reference}'] = df.apply(lambda row: 1 if is_between(0.75, row[auxiliary_percentile_column], 0.85) & (row[auxiliary_percentile_lag_column] < 0.75)
                                      else 0
                                      , axis=1)
    
    df[f'change_{reference}_above75'] = df.apply(lambda row: 1 if (row[auxiliary_percentile_column] >= 0.75) & (row[auxiliary_percentile_lag_column] < 0.75)
                                      else 0
                                      , axis=1)

In [0]:
create_variable_change(column='temp_mean_weekly_mean', reference='temperature')
create_variable_change(column='precipitation_weekly_sum', reference='precipitation')
create_variable_change(column='humidity_weekly_mean', reference='humidity')

In [0]:
# Check
df

In [0]:
# Check
print(df[['change_temperature', 'change_temperature_above75']].value_counts())
print(df[['change_precipitation', 'change_precipitation_above75']].value_counts())
print(df[['change_humidity', 'change_humidity_above75']].value_counts())

##### 1.3 Interaction

- interaction_temperature_precipitation (0 or 1 - when change_temperature and change_precipitation were equal to 1, but humidity was 0)
- interaction_temperature_humidity (0 or 1 - when change_temperature and change_humidity were equal to 1, but precipitation was 0)
- interaction_precipitation_humidity (0 or 1 - when change_precipitation and change_humidity were equal to 1, but temperature was 0)
- interaction_temperature_precipitation_humidity (0 or 1 - when change_temperature, interaction_temperature_humidity and change_humidity were equal to 1)

In [0]:
df['interaction_temperature_precipitation'] = df.apply(lambda row: 1 if (row['change_temperature'] == 1) & (row['change_precipitation'] == 1) & (row['change_humidity'] == 0)
        else 0
        , axis=1)

df['interaction_temperature_humidity'] = df.apply(lambda row: 1 if (row['change_temperature'] == 1) & (row['change_precipitation'] == 0) & (row['change_humidity'] == 1)
        else 0
        , axis=1)

df['interaction_precipitation_humidity'] = df.apply(lambda row: 1 if (row['change_temperature'] == 0) & (row['change_precipitation'] == 1) & (row['change_humidity'] == 1)
        else 0
        , axis=1)

df['interaction_temperature_precipitation_humidity'] = df.apply(lambda row: 1 if (row['change_temperature'] == 1) & (row['change_precipitation'] == 1) & (row['change_humidity'] == 1)
        else 0
        , axis=1)

In [0]:
# Check
df[['change_temperature', 'change_precipitation', 'change_humidity', 'interaction_temperature_precipitation_humidity']].value_counts()

In [0]:
df['interaction_temperature_precipitation_above75'] = df.apply(lambda row: 1 if (row['change_temperature_above75'] == 1) & (row['change_precipitation_above75'] == 1) & (row['change_humidity_above75'] == 0)
        else 0
        , axis=1)

df['interaction_temperature_humidity_above75'] = df.apply(lambda row: 1 if (row['change_temperature_above75'] == 1) & (row['change_precipitation_above75'] == 0) & (row['change_humidity_above75'] == 1)
        else 0
        , axis=1)

df['interaction_precipitation_humidity_above75'] = df.apply(lambda row: 1 if (row['change_temperature_above75'] == 0) & (row['change_precipitation_above75'] == 1) & (row['change_humidity_above75'] == 1)
        else 0
        , axis=1)

df['interaction_temperature_precipitation_humidity_above75'] = df.apply(lambda row: 1 if (row['change_temperature_above75'] == 1) & (row['change_precipitation_above75'] == 1) & (row['change_humidity_above75'] == 1)
        else 0
        , axis=1)

In [0]:
# Check
df[['change_temperature_above75', 'change_precipitation_above75', 'change_humidity_above75', 'interaction_temperature_precipitation_humidity_above75']].value_counts(dropna=False, sort=False)

In [0]:
# Check
df

##### 2.4 Extreme Weather

- extreme_weather_temperature (0 or 1 - percentile above 0.95)
- extreme_weather_rain (0 or 1 - percentile above 0.95)
- extreme_weather_humidity (0 or 1 - percentile above 0.95)

In [0]:
def create_extreme_weather(column, reference):
    global df

    new_column_name = f'extreme_weather_{reference}'

    auxiliary_percentile_column = f'percentile_{column}'
    
    df[new_column_name] = df.apply(lambda row: 1 if (row[auxiliary_percentile_column] > 0.95)
        else 0
        , axis=1)

In [0]:
create_extreme_weather(column='temp_mean_weekly_mean', reference='temperature')
create_extreme_weather(column='precipitation_weekly_sum', reference='precipitation')
create_extreme_weather(column='humidity_weekly_mean', reference='humidity')

In [0]:
# Check
df

In [0]:
#Check
print(df[df['percentile_temp_mean_weekly_mean'] > 0.95].shape[0])
print(df[df['percentile_precipitation_weekly_sum'] > 0.95].shape[0])
print(df[df['percentile_humidity_weekly_mean'] > 0.95].shape[0])

In [0]:
#Check
print(df['extreme_weather_temperature'].value_counts(dropna=False, sort=False))
print(df['extreme_weather_precipitation'].value_counts(dropna=False, sort=False))
print(df['extreme_weather_humidity'].value_counts(dropna=False, sort=False))

##### 1.5 Season

- season_summer (0 or 1 - summer criterion)
- season_spring (0 or 1 - spring criterion)
- season_autumn (0 or 1 - autumn criterion)
- season_winter (0 or 1 - winter criterion)

In [0]:
def create_summer_season(date):
    year = date.year
    if pd.Timestamp(f'{year}-12-21') <= date or date < pd.Timestamp(f'{year}-03-21'):
        return 1
    else:
        return 0
    
def create_autumn_season(date):
    year = date.year
    if pd.Timestamp(f'{year}-03-21') <= date < pd.Timestamp(f'{year}-06-21'):
        return 1
    else:
        return 0
    
def create_winter_season(date):
    year = date.year
    if pd.Timestamp(f'{year}-06-21') <= date < pd.Timestamp(f'{year}-09-23'):
        return 1
    else:
        return 0
    
def create_spring_season(date):
    year = date.year
    if pd.Timestamp(f'{year}-09-23') <= date < pd.Timestamp(f'{year}-12-21'):
        return 1
    else:
        return 0

In [0]:
df['season_summer'] = df['Data Medicao'].apply(create_summer_season)
df['season_autumn'] = df['Data Medicao'].apply(create_autumn_season)
df['season_winter'] = df['Data Medicao'].apply(create_winter_season)
df['season_spring'] = df['Data Medicao'].apply(create_spring_season)
df

In [0]:
# Check
df[['season_summer', 'season_autumn', 'season_winter', 'season_spring']].value_counts(dropna=False, sort=False)

##### 1.5 Temperature Category

- temperature_category_quantile_01 (0 or 1 - temperatures in percentile 0% - 25%)
- temperature_category_quantile_02 (0 or 1 - temperatures in percentile 26% - 50%)
- temperature_category_quantile_03 (0 or 1 - temperatures in percentile 51% - 75%)
- temperature_category_quantile_04 (0 or 1 - temperatures in percentile 76% - 100%)

In [0]:
df['temperature_category_quantile_01'] = df['percentile_temp_mean_weekly_mean'].apply(lambda x: 1 if x <= 0.25 else 0)
df['temperature_category_quantile_02'] = df['percentile_temp_mean_weekly_mean'].apply(lambda x: 1 if 0.25 < x <= 0.5 else 0)
df['temperature_category_quantile_03'] = df['percentile_temp_mean_weekly_mean'].apply(lambda x: 1 if 0.5 < x <= 0.75 else 0)
df['temperature_category_quantile_04'] = df['percentile_temp_mean_weekly_mean'].apply(lambda x: 1 if x > 0.75 else 0)

In [0]:
#Check
print(df[['temperature_category_quantile_01', 'percentile_temp_mean_weekly_mean']].value_counts(dropna=False, sort=False))
print(df[['temperature_category_quantile_02', 'percentile_temp_mean_weekly_mean']].value_counts(dropna=False, sort=False))
print(df[['temperature_category_quantile_03', 'percentile_temp_mean_weekly_mean']].value_counts(dropna=False, sort=False))
print(df[['temperature_category_quantile_04', 'percentile_temp_mean_weekly_mean']].value_counts(dropna=False, sort=False))

##### 2.6 Precipitation Occurrence

- precipitation_occurrence (0 or 1 - criterion: rainfall occurred)

In [0]:
df['precipitation_occurrence'] = (df['precipitation_weekly_sum'] > 0).astype(int)

In [0]:
#Check
df[['precipitation_weekly_sum', 'precipitation_occurrence']]

##### 2.7 Rainfall

- sequential_rain (sum of the number of sequential rainy days)
- rain_days (sum of the number of rainy days)

In [0]:
df.rename(columns={'precipitation_days_with_rain': 'rain_days', 'precipitation_consecutive_days_with_rain': 'sequential_rain'}, inplace=True)
df

In [0]:
# Converting date to week format, as used in the other hospitalization and WB access tables
df.rename(columns={'Data Medicao': 'date'}, inplace=True)
df['date'] = df['date'].dt.strftime('%Y-%U')
display(df)

In [0]:
# Export data
df.to_csv('Data Climate/Data Climate Processed/ready_validated_stations_filled_gap30_all_features.csv', index=False)